# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

> 주 제출물은 실행 완료 Notebook `chapter05/chapter05.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026-09-10
- 최종 제출 URL: https://github.com/sangjae-lee97/kant-axagent-study/blob/main/llm-data-analysis-course/chapter05/chapter05.ipynb

## 1. 전처리 전 상태 기록
- 각 데이터 shape: 
(150, 6)
(100, 4)
(300, 5)
(764, 5)
- 결측치: 
missing: 0
missing: 0
missing: 0
missing: 0
- 전체 중복:
duplicated rows: 0
duplicated rows: 0
duplicated rows: 0
duplicated rows: 0
- 주요 ID 중복:
duplicated customer_id: 0
duplicated product_id: 0
duplicated order_id: 0
duplicated order_item_id: 0
- 타입 문제 후보:
order_date, signup_date

![전처리 전 상태](images/step01_before.png)

In [58]:
from course_utils.paths import get_project_root, get_data_dir
import pandas as pd

print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())

프로젝트 루트 : C:\dev\kant-axagent-study\llm-data-analysis-course
데이터 폴더 : C:\dev\kant-axagent-study\llm-data-analysis-course\data


In [59]:
RAW_DIR = get_project_root() / "data" / "raw"
PROCESSED_DIR = get_project_root() / "data" / "processed"
REPORT_DIR = get_project_root() / "reports"

In [60]:
# 데이터 프레임으로 불러오기
customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")

In [61]:
# 각 데이터 프레임 shape 확인
print(customers.shape)
print(products.shape)
print(orders.shape)
print(order_items.shape)

(150, 6)
(100, 4)
(300, 5)
(764, 5)


In [62]:
# 전처리 전 상태 기록
raw_data = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items
}

id_columns = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id"
}

for name, df in raw_data.items():
    print(name)
    print("shape:", df.shape)
    print("missing:", int(df.isna().sum().sum()))
    print("duplicated rows:", int(df.duplicated().sum()))

    id_col = id_columns[name]

    print(
        f"duplicated {id_col}:",
        int(df[id_col].duplicated().sum())
    )

    print()

customers
shape: (150, 6)
missing: 0
duplicated rows: 0
duplicated customer_id: 0

products
shape: (100, 4)
missing: 0
duplicated rows: 0
duplicated product_id: 0

orders
shape: (300, 5)
missing: 0
duplicated rows: 0
duplicated order_id: 0

order_items
shape: (764, 5)
missing: 0
duplicated rows: 0
duplicated order_item_id: 0



In [63]:
print("id_col:", id_col)
print("id_col type:", type(id_col))
print("df type:", type(df))

id_col: order_item_id
id_col type: <class 'str'>
df type: <class 'pandas.DataFrame'>


### 결과 관찰
데이터 프레임으로 변환하여 간단히 데이터를 확인 하였지만 문제 없음
### 나의 해석과 판단
결측지, 중복된 행, 중복된 id 없음 확인 완료하여 데이터로 쓸 수 있을거 같음

### 업무·분석적 의미
기본적인 결측치, 중복된 정보 있는지 확인 완료
### 한계와 추가 확인 사항
결측치와 중복행은 없지만 타입이 잘못 되었는지 확인 불가능

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: dtypes 확인 후 문자열 타입은 결측치가 아닌것은 공백 제거, 결측치 인것은 NA로 변환 완료
- 숫자 변환 대상과 실패 건수:
[customers]
customer_id 변환 실패: 0
age 변환 실패: 0

[products]
product_id 변환 실패: 0
price 변환 실패: 0

[orders]
order_id 변환 실패: 0
customer_id 변환 실패: 0

[order_items]
order_item_id 변환 실패: 0
order_id 변환 실패: 0
product_id 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0
- 날짜 변환 대상과 실패 건수:
signup_date 변환 실패: 0
order_date 변환 실패: 0
- 결측 처리 기준: 
수치형 컬럼은 데이터 분포의 영향을 덜 받도록 중앙값으로 대체하고, 문자열형 컬럼은 빈 문자열을 pd.NA로 통일하였다. 날짜형 컬럼은 임의의 날짜로 대체하지 않고 결측 상태를 유지하였다.

![타입과 결측 처리](images/step02_cleaning.png)

In [64]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

In [65]:
print(customers_clean.dtypes)
print(products_clean.dtypes)
print(orders_clean.dtypes)
print(order_items_clean.dtypes)

customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object
product_id      int64
product_name      str
category          str
price           int64
dtype: object
order_id          int64
customer_id       int64
order_date          str
payment_method      str
order_status        str
dtype: object
order_item_id    int64
order_id         int64
product_id       int64
quantity         int64
unit_price       int64
dtype: object


In [66]:
# =========================
# customers_clean
# =========================

customers_clean["name"] = customers_clean["name"].where(
    customers_clean["name"].isna(),
    customers_clean["name"].astype(str).str.strip(),
)

customers_clean["name"] = customers_clean["name"].replace(
    "",
    pd.NA,
)


customers_clean["gender"] = customers_clean["gender"].where(
    customers_clean["gender"].isna(),
    customers_clean["gender"].astype(str).str.strip(),
)

customers_clean["gender"] = customers_clean["gender"].replace(
    "",
    pd.NA,
)


customers_clean["city"] = customers_clean["city"].where(
    customers_clean["city"].isna(),
    customers_clean["city"].astype(str).str.strip(),
)

customers_clean["city"] = customers_clean["city"].replace(
    "",
    pd.NA,
)


customers_clean["signup_date"] = customers_clean["signup_date"].where(
    customers_clean["signup_date"].isna(),
    customers_clean["signup_date"].astype(str).str.strip(),
)

customers_clean["signup_date"] = customers_clean["signup_date"].replace(
    "",
    pd.NA,
)

In [67]:
# =========================
# products_clean
# =========================

products_clean["product_name"] = products_clean["product_name"].where(
    products_clean["product_name"].isna(),
    products_clean["product_name"].astype(str).str.strip(),
)

products_clean["product_name"] = products_clean["product_name"].replace(
    "",
    pd.NA,
)


products_clean["category"] = products_clean["category"].where(
    products_clean["category"].isna(),
    products_clean["category"].astype(str).str.strip(),
)

products_clean["category"] = products_clean["category"].replace(
    "",
    pd.NA,
)

In [68]:
# =========================
# orders_clean
# =========================

orders_clean["order_date"] = orders_clean["order_date"].where(
    orders_clean["order_date"].isna(),
    orders_clean["order_date"].astype(str).str.strip(),
)

orders_clean["order_date"] = orders_clean["order_date"].replace(
    "",
    pd.NA,
)


orders_clean["order_status"] = orders_clean["order_status"].where(
    orders_clean["order_status"].isna(),
    orders_clean["order_status"].astype(str).str.strip(),
)

orders_clean["order_status"] = orders_clean["order_status"].replace(
    "",
    pd.NA,
)


orders_clean["payment_method"] = orders_clean["payment_method"].where(
    orders_clean["payment_method"].isna(),
    orders_clean["payment_method"].astype(str).str.strip(),
)

orders_clean["payment_method"] = orders_clean["payment_method"].replace(
    "",
    pd.NA,
)

In [69]:
# customers_clean

converted_customer_id = pd.to_numeric(
    customers_clean["customer_id"],
    errors="coerce"
)

customer_id_fail_count = (
    converted_customer_id.isna()
    & customers_clean["customer_id"].notna()
).sum()

print("customer_id 변환 실패:", int(customer_id_fail_count))

customer_id 변환 실패: 0


In [70]:
converted_age = pd.to_numeric(
    customers_clean["age"],
    errors="coerce"
)

age_fail_count = (
    converted_age.isna()
    & customers_clean["age"].notna()
).sum()

print("age 변환 실패:", int(age_fail_count))

age 변환 실패: 0


In [71]:
# products_clean

converted_product_id = pd.to_numeric(
    products_clean["product_id"],
    errors="coerce"
)

product_id_fail_count = (
    converted_product_id.isna()
    & products_clean["product_id"].notna()
).sum()

print("product_id 변환 실패:", int(product_id_fail_count))

product_id 변환 실패: 0


In [72]:
converted_price = pd.to_numeric(
    products_clean["price"],
    errors="coerce"
)

price_fail_count = (
    converted_price.isna()
    & products_clean["price"].notna()
).sum()

print("price 변환 실패:", int(price_fail_count))

price 변환 실패: 0


In [73]:
# orders_clean

converted_order_id = pd.to_numeric(
    orders_clean["order_id"],
    errors="coerce"
)

order_id_fail_count = (
    converted_order_id.isna()
    & orders_clean["order_id"].notna()
).sum()

print("order_id 변환 실패:", int(order_id_fail_count))

order_id 변환 실패: 0


In [74]:
converted_customer_id_orders = pd.to_numeric(
    orders_clean["customer_id"],
    errors="coerce"
)

customer_id_orders_fail_count = (
    converted_customer_id_orders.isna()
    & orders_clean["customer_id"].notna()
).sum()

print("customer_id 변환 실패:", int(customer_id_orders_fail_count))

customer_id 변환 실패: 0


In [75]:
# order_items_clean

converted_order_item_id = pd.to_numeric(
    order_items_clean["order_item_id"],
    errors="coerce"
)

order_item_id_fail_count = (
    converted_order_item_id.isna()
    & order_items_clean["order_item_id"].notna()
).sum()

print("order_item_id 변환 실패:", int(order_item_id_fail_count))

order_item_id 변환 실패: 0


In [76]:
converted_order_id_items = pd.to_numeric(
    order_items_clean["order_id"],
    errors="coerce"
)

order_id_items_fail_count = (
    converted_order_id_items.isna()
    & order_items_clean["order_id"].notna()
).sum()

print("order_id 변환 실패:", int(order_id_items_fail_count))

order_id 변환 실패: 0


In [77]:
converted_quantity = pd.to_numeric(
    order_items_clean["quantity"],
    errors="coerce"
)

quantity_fail_count = (
    converted_quantity.isna()
    & order_items_clean["quantity"].notna()
).sum()

print("quantity 변환 실패:", int(quantity_fail_count))

quantity 변환 실패: 0


In [78]:
converted_unit_price = pd.to_numeric(
    order_items_clean["unit_price"],
    errors="coerce"
)

unit_price_fail_count = (
    converted_unit_price.isna()
    & order_items_clean["unit_price"].notna()
).sum()

print("unit_price 변환 실패:", int(unit_price_fail_count))

unit_price 변환 실패: 0


In [95]:
converted_signup_date = pd.to_datetime(
    customers_clean["signup_date"],
    errors="coerce"
)

signup_date_fail_count = (
    converted_signup_date.isna()
    & customers_clean["signup_date"].notna()
).sum()

print("signup_date 변환 실패:", int(signup_date_fail_count))

customers_clean["signup_date"] = converted_signup_date

signup_date 변환 실패: 0


In [96]:
converted_order_date = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

order_date_fail_count = (
    converted_order_date.isna()
    & orders_clean["order_date"].notna()
).sum()

print("order_date 변환 실패:", int(order_date_fail_count))

orders_clean["order_date"] = converted_order_date

order_date 변환 실패: 0


In [81]:
age_median = customers_clean["age"].median()

customers_clean["age"] = (
    customers_clean["age"].fillna(age_median)
)

In [82]:

customers_clean["city"] = (
    customers_clean["city"].fillna("Unknown")
)


### 결과 관찰

수치형 결측치는 중앙값으로 대체하도록 처리했으며, 현재 데이터에서는 실제 결측치가 없어 값의 변화는 없었다.

### 나의 해석과 판단

중앙값은 이상치의 영향을 비교적 덜 받기 때문에 수치형 결측치 대체 기준으로 선택했다. 다만 값을 대체하면 실제 원래 값이 아닌 추정값이 들어가므로 정보 왜곡 가능성이 있다.

### 업무·분석적 의미

결측치 때문에 분석이나 계산이 중단되는 것을 방지하고, 데이터 활용도를 높일 수 있다.

### 한계와 추가 확인 사항

결측치가 많아질 경우 중앙값 대체가 적절한지 다시 확인해야 하며, 컬럼별 업무 의미에 따라 다른 처리 기준이 필요할 수 있다.


## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 전체 테이블에서 완전 중복 행과 주요 ID 중복이 모두 0건으로 확인되어 제거한 데이터는 없음.

- 표준화한 범주값: `gender`, `category`, `payment_method`, `order_status`의 범주값을 확인했으며, 현재 데이터에서는 별도 표준화가 필요한 값이 발견되지 않음.

- 허용값 밖 값: `gender`, `category`, `payment_method`, `order_status` 모두 허용값 밖 데이터가 0건으로 확인됨.

- 이상값 후보: age는 0 미만 또는 120 초과, price, quantity, unit_price는 0 이하를 이상값 후보 기준으로 확인했으며, 해당하는 데이터는 발견되지 않았다.


![중복 범주 이상값](images/step03_rules.png)

In [83]:
# 1. 제거/보류할 중복 확인

print("customers 중복:", customers_clean.duplicated().sum())
print("products 중복:", products_clean.duplicated().sum())
print("orders 중복:", orders_clean.duplicated().sum())
print("order_items 중복:", order_items_clean.duplicated().sum())

customers 중복: 0
products 중복: 0
orders 중복: 0
order_items 중복: 0


In [84]:
print(
    "customer_id 중복:",
    customers_clean["customer_id"].duplicated().sum()
)

print(
    "product_id 중복:",
    products_clean["product_id"].duplicated().sum()
)

print(
    "order_id 중복:",
    orders_clean["order_id"].duplicated().sum()
)

print(
    "order_item_id 중복:",
    order_items_clean["order_item_id"].duplicated().sum()
)

customer_id 중복: 0
product_id 중복: 0
order_id 중복: 0
order_item_id 중복: 0


In [85]:
# 2. 표준화할 범주값 확인

print(customers_clean["gender"].value_counts())
print()

print(customers_clean["city"].value_counts())
print()

print(products_clean["category"].value_counts())
print()

print(orders_clean["payment_method"].value_counts())
print()

print(orders_clean["order_status"].value_counts())

gender
F    84
M    66
Name: count, dtype: int64

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

category
스포츠     19
전자기기    17
생활용품    16
뷰티      16
도서      14
패션      11
식품       7
Name: count, dtype: int64

payment_method
kakao_pay        79
naver_pay        77
bank_transfer    74
card             70
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64


In [86]:
# gender 허용값 검사

allowed_gender = ["M", "F"]

invalid_gender = customers_clean[
    ~customers_clean["gender"].isin(allowed_gender)
]

print("gender 허용값 밖:", len(invalid_gender))

gender 허용값 밖: 0


In [87]:
# category 허용값 검사

allowed_category = [
    "도서",
    "뷰티",
    "생활용품",
    "스포츠",
    "식품",
    "전자기기",
    "패션",
]

invalid_category = products_clean[
    ~products_clean["category"].isin(allowed_category)
]

print("category 허용값 밖:", len(invalid_category))

category 허용값 밖: 0


In [88]:
# payment_method 허용값 검사

allowed_payment_method = [
    "bank_transfer",
    "card",
    "kakao_pay",
    "naver_pay",
]

invalid_payment_method = orders_clean[
    ~orders_clean["payment_method"].isin(allowed_payment_method)
]

print(
    "payment_method 허용값 밖:",
    len(invalid_payment_method)
)

payment_method 허용값 밖: 0


In [89]:
# order_status 허용값 검사

allowed_order_status = [
    "cancelled",
    "completed",
    "refunded",
]

invalid_order_status = orders_clean[
    ~orders_clean["order_status"].isin(allowed_order_status)
]

print(
    "order_status 허용값 밖:",
    len(invalid_order_status)
)

order_status 허용값 밖: 0


In [90]:
# age 이상값 후보
customers_clean.loc[
    (customers_clean["age"] < 0)
    | (customers_clean["age"] > 120)
]


,customer_id,name,gender,age,city,signup_date


In [91]:
# price 이상값 후보
products_clean.loc[
    products_clean["price"] <= 0
]

,product_id,product_name,category,price


In [92]:
# quantity 이상값 후보
order_items_clean.loc[
    order_items_clean["quantity"] <= 0
]

,order_item_id,order_id,product_id,quantity,unit_price


In [93]:
# unit_price 이상값 후보
order_items_clean.loc[
    order_items_clean["unit_price"] <= 0
]

,order_item_id,order_id,product_id,quantity,unit_price



### 결과 관찰

전체 테이블에서 완전 중복 행과 주요 ID 중복은 확인되지 않았다. 범주형 컬럼에서도 허용값 밖 데이터는 없었으며, `age`는 0 미만 또는 120 초과, `price`, `quantity`, `unit_price`는 0 이하 기준으로 확인했을 때 이상값 후보가 발견되지 않았다.

### 나의 해석과 판단

이상값처럼 보이는 값도 실제 업무에서는 정상 데이터일 수 있으므로 자동 삭제하지 않고 후보로 검토하는 방식이 적절하다고 판단했다. 자동 삭제하면 실제로 의미 있는 데이터까지 제거되어 정보 손실이 발생할 수 있다.

### 업무·분석적 의미

중복, 범주값, 이상값 후보를 사전에 확인하면 잘못된 데이터로 인해 분석 결과가 왜곡되는 것을 줄일 수 있다.

### 한계와 추가 확인 사항

현재는 업무적으로 명확한 범위 기준만 적용


## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0
- 없는 `order_id`: order_month, order_dayofweek
- 없는 `product_id`: 0
- 생성한 파생 컬럼: order_month, order_dayofweek,line_total
- `line_total` 검증: 완료

![관계와 파생 컬럼 검증](images/step04_validation.png)

In [97]:
orders_clean["order_month"] = (
    orders_clean["order_date"]
    .dt.to_period("M")
    .astype(str)
)

In [98]:
orders_clean["order_dayofweek"] = (
    orders_clean["order_date"].dt.day_name()
)

In [99]:
order_items_clean["line_total"] = (
    order_items_clean["quantity"]
    * order_items_clean["unit_price"]
)

In [100]:
# 라인 토탈 검증
line_total_check = (
    order_items_clean["line_total"]
    == (
        order_items_clean["quantity"]
        * order_items_clean["unit_price"]
    )
)

print("line_total 정상:", line_total_check.sum())
print("line_total 불일치:", (~line_total_check).sum())

line_total 정상: 764
line_total 불일치: 0


### 결과 관찰

전처리 후 `customer_id`, `order_id`, `product_id`의 관계를 다시 확인했으며, 참조되지 않는 ID는 발견되지 않았다. 또한 `order_month`, `order_dayofweek`, `line_total` 파생 컬럼을 생성하고 계산 결과를 검증했다.

### 나의 해석과 판단

전처리 과정에서 값이 변환되거나 결측 처리, 행 삭제 등이 발생하면 기존 PK/FK 관계가 깨질 수 있기 때문에 관계를 다시 검증해야 한다고 판단했다. 특히 다른 테이블에 존재하지 않는 ID가 생기면 이후 병합이나 분석 결과가 잘못될 수 있다.


## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 동일 | 동일 | 삭제한 행 없음 |
| 결측 | 0 | 0 | 추가 결측 없음 |
| 중복 | 0 | 0 | 중복 제거 대상 없음 |
| 변환 실패 | 0 | 0 | 숫자·날짜 변환 실패 없음 |

![전처리 전후 비교](images/step05_before_after.png)

In [105]:
# 행 수
print("customers 행 수:", len(customers), "->", len(customers_clean))
print("products 행 수:", len(products), "->", len(products_clean))
print("orders 행 수:", len(orders), "->", len(orders_clean))
print("order_items 행 수:", len(order_items), "->", len(order_items_clean))

customers 행 수: 150 -> 150
products 행 수: 100 -> 100
orders 행 수: 300 -> 300
order_items 행 수: 764 -> 764


In [106]:
# 결측
print(
    "customers 결측:",
    int(customers.isna().sum().sum()),
    "->",
    int(customers_clean.isna().sum().sum())
)

print(
    "products 결측:",
    int(products.isna().sum().sum()),
    "->",
    int(products_clean.isna().sum().sum())
)

print(
    "orders 결측:",
    int(orders.isna().sum().sum()),
    "->",
    int(orders_clean.isna().sum().sum())
)

print(
    "order_items 결측:",
    int(order_items.isna().sum().sum()),
    "->",
    int(order_items_clean.isna().sum().sum())
)


customers 결측: 0 -> 0
products 결측: 0 -> 0
orders 결측: 0 -> 0
order_items 결측: 0 -> 0


In [107]:
# 중복
print(
    "customers 중복:",
    int(customers.duplicated().sum()),
    "->",
    int(customers_clean.duplicated().sum())
)

print(
    "products 중복:",
    int(products.duplicated().sum()),
    "->",
    int(products_clean.duplicated().sum())
)

print(
    "orders 중복:",
    int(orders.duplicated().sum()),
    "->",
    int(orders_clean.duplicated().sum())
)

print(
    "order_items 중복:",
    int(order_items.duplicated().sum()),
    "->",
    int(order_items_clean.duplicated().sum())
)

customers 중복: 0 -> 0
products 중복: 0 -> 0
orders 중복: 0 -> 0
order_items 중복: 0 -> 0


### 결과 관찰

전처리 전후의 행 수, 결측치, 중복 건수를 비교했으며 큰 변화는 없었다. 숫자와 날짜 변환 실패도 발생하지 않았다.

### 나의 해석과 판단

행 수가 유지되고 값 분포에도 큰 변화가 없었기 때문에 전처리가 원본 데이터를 과도하게 변경하지 않고 의도한 범위에서 수행되었다고 판단했다. 불필요한 삭제나 대체가 없었기 때문에 정보 손실도 크지 않다.

### 업무·분석적 의미

전처리 전후를 비교하면 데이터 정제 과정에서 원본 정보가 예상치 않게 손실되거나 왜곡되지 않았는지 확인할 수 있다.

### 한계와 추가 확인 사항

현재는 행 수, 결측, 중복, 변환 실패 위주로 비교했기 때문에 필요하면 평균, 중앙값, 범주별 빈도 등 값 분포도 추가로 비교할 수 있다.

## 6. 재현 가능한 전처리 확인

* 생성된 clean CSV: `customers_clean.csv`, `products_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`

* 생성된 요약 보고서: `reports/ch05_preprocessing_summary.md`

* `python scripts/preprocess_data.py` 재실행 결과: 전처리가 정상적으로 완료되었으며, 전처리 전후 행 수는 동일하게 유지되었다. `orders`에는 `order_month`, `order_dayofweek`가 추가되어 컬럼 수가 5개에서 7개로 증가했고, `order_items`에는 `line_total`이 추가되어 컬럼 수가 5개에서 6개로 증가했다. 중복 및 PK 중복은 모두 0건이었고, 파일 간 FK 관계 점검에서도 유효하지 않은 값은 모두 0건이었다.

* 재실행 후 달라진 점: 원본 행 수에는 변화가 없었으며, 의도한 파생 컬럼만 추가되었다. 전처리 결과 파일과 요약 보고서도 정상적으로 다시 생성되었다.


![재실행 결과](images/step06_reproducible.png)


In [113]:
customers_clean.to_csv(
    PROCESSED_DIR/"customers_clean.csv",
    index=False
)

products_clean.to_csv(
   PROCESSED_DIR/"products_clean.csv",
    index=False
)

orders_clean.to_csv(
    PROCESSED_DIR/"orders_clean.csv",
    index=False
)

order_items_clean.to_csv(
    PROCESSED_DIR/"order_items_clean.csv",
    index=False
)

## 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지

1. 원본 데이터는 유지하고 clean 데이터에서만 수정한다.
2. 결측치와 이상값은 자동 삭제하지 않고 의미를 확인한 뒤 처리한다.
3. 전처리 후에는 중복, 타입, PK/FK 관계를 다시 검증한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유

이상값 자동 삭제가 가장 위험하다고 생각했다. 실제로는 정상적인 값일 수 있어 의미 있는 데이터가 손실될 수 있기 때문이다.

### 다음 EDA에서 특히 주의할 데이터 특성

가격, 수량, 연령 등의 분포와 범주별 데이터 개수 차이를 주의해서 확인할 필요가 있다.

### 현재 전처리의 한계

기본적인 오류와 관계 검증 위주로 처리했기 때문에, 통계적 이상값이나 세부적인 데이터 분포는 EDA에서 추가 확인이 필요하다.

## 최종 제출 체크

* [x] raw 원본을 덮어쓰지 않았습니다.
* [x] 처리 전/후 비교를 남겼습니다.
* [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
* [x] clean CSV와 재실행 결과를 확인했습니다.
* [x] 개인정보/Secret이 없습니다.
* [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
* [x] 최종 Notebook 파일 URL을 제출합니다.
